# PART B — MACHINE LEARNING

Distributed Spark ML regression for predicting FHVHV `trip_time`. Run Part A first; this notebook consumes only its Silver Parquet output.

The feature set contains only pickup-time-safe fields prepared by Part A.

In [ ]:
# Configuration — Part A must have written this exact Silver contract first
SPARK_MASTER = 'spark://spark-master:7077'
PROCESSED_DATA_PATH = 's3a://silver/fhvhv/2025/processed_trip_time_features'
PREDICTIONS_BASE_PATH = 's3a://gold/fhvhv/2025/trip_time_predictions'
MODEL_BASE_PATH = 's3a://gold/fhvhv/2025/trip_time_models'
LOG_DIR = '/workspace/logs'
TEST_MODE = True
TEST_FRACTION = 0.002
SHUFFLE_PARTITIONS = 24 if TEST_MODE else 64
LABEL_COL = 'trip_time'
CATEGORICAL_COLS = ['hvfhs_license_num', 'PULocationID', 'DOLocationID', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']
NUMERIC_COLS = ['trip_miles', 'pickup_hour', 'pickup_day_of_week', 'pickup_month', 'is_weekend']
print(f'TEST_MODE={TEST_MODE}; master={SPARK_MASTER}; processed input={PROCESSED_DATA_PATH}')

In [ ]:
# Persistent console + file logging; no credentials are placed in this notebook or log.
import logging, os, sys, time, platform
from pathlib import Path
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)
LOG_PATH = os.path.join(LOG_DIR, 'fhvhv_part_b_ml.log')
logger = logging.getLogger('fhvhv.part_b')
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
for handler in (logging.FileHandler(LOG_PATH, encoding='utf-8'), logging.StreamHandler(sys.stdout)):
    handler.setFormatter(formatter)
    logger.addHandler(handler)
benchmarks, stage_started = {}, {}
pipeline_started = time.perf_counter()
def start_stage(name):
    stage_started[name] = time.perf_counter(); logger.info('[ML] %s started', name)
def end_stage(name):
    elapsed = time.perf_counter() - stage_started[name]; benchmarks[name] = elapsed
    logger.info('[ML] %s completed in %.2f sec', name, elapsed); return elapsed
logger.info('[SYSTEM] Python=%s platform=%s TEST_MODE=%s', platform.python_version(), platform.platform(), TEST_MODE)
logger.info('[SYSTEM] Persistent log=%s', LOG_PATH)

## Connect to Spark and load the Part A Silver dataset

In [ ]:
start_stage('Spark initialization')
from pyspark.sql import SparkSession
from pyspark import StorageLevel
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
spark = (SparkSession.builder.appName('FHVHV-Part-B-Trip-Time-Regression').master(SPARK_MASTER)
    .config('spark.sql.shuffle.partitions', SHUFFLE_PARTITIONS)
    .config('spark.default.parallelism', SHUFFLE_PARTITIONS)
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.executor.memory', '2g')
    .config('spark.executor.cores', '4')
    .config('spark.executor.memoryOverhead', '512m')
    .config('spark.dynamicAllocation.enabled', 'false').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
sc = spark.sparkContext
if sc.master.startswith('local'):
    raise RuntimeError(f'[CLUSTER] Local Spark fallback is forbidden; actual master={sc.master}')
if not sc._jsc.hadoopConfiguration().get('fs.s3a.impl'):
    raise RuntimeError('[DATA_ACCESS] S3A is not configured. Configure Spark/Kubernetes environment without adding credentials to this notebook.')
logger.info('[CLUSTER] application=%s Spark=%s master=%s defaultParallelism=%s', sc.applicationId, spark.version, sc.master, sc.defaultParallelism)
end_stage('Spark initialization')

start_stage('Processed data loading')
logger.info('[DATA_LOADING] Reading processed Parquet from %s', PROCESSED_DATA_PATH)
processed_df = spark.read.parquet(PROCESSED_DATA_PATH)
expected_columns = [LABEL_COL] + CATEGORICAL_COLS + NUMERIC_COLS
missing_columns = [column for column in expected_columns if column not in processed_df.columns]
if missing_columns: raise ValueError(f'[DATA_LOADING] Part A output misses required columns: {missing_columns}')
if TEST_MODE:
        processed_df = processed_df.sample(fraction=TEST_FRACTION, seed=42)
        logger.warning('[DATA_LOADING] TEST_MODE enabled: sample=%.0f%%  est_rows=%s', TEST_FRACTION*100, int(TEST_FRACTION*243000000))
processed_df = processed_df.repartition(SHUFFLE_PARTITIONS).persist(StorageLevel.MEMORY_AND_DISK)
logger.info('[DATA_LOADING] partitions=%s rows=%s', processed_df._jdf.rdd().getNumPartitions(), processed_df.count())
processed_df.show(10, truncate=False)
end_stage('Processed data loading')

## Train/test split and train-only preprocessing

In [ ]:
start_stage('Train test split')
train_df, test_df = processed_df.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df = test_df.persist(StorageLevel.MEMORY_AND_DISK)
logger.info('[ML] train_rows=%s test_rows=%s', train_df.count(), test_df.count())
end_stage('Train test split')

start_stage('ML preprocessing fit')
categorical_indexers = [StringIndexer(inputCol=column, outputCol=f'{column}_index', handleInvalid='keep') for column in CATEGORICAL_COLS]
encoder = OneHotEncoder(inputCols=[f'{column}_index' for column in CATEGORICAL_COLS], outputCols=[f'{column}_ohe' for column in CATEGORICAL_COLS], handleInvalid='keep')
numeric_assembler = VectorAssembler(inputCols=NUMERIC_COLS, outputCol='numeric_features_raw', handleInvalid='keep')
scaler = StandardScaler(inputCol='numeric_features_raw', outputCol='numeric_features_scaled', withMean=False, withStd=True)
feature_assembler = VectorAssembler(inputCols=['numeric_features_scaled'] + [f'{column}_ohe' for column in CATEGORICAL_COLS], outputCol='features', handleInvalid='keep')
preprocessing_pipeline = Pipeline(stages=categorical_indexers + [encoder, numeric_assembler, scaler, feature_assembler])
preprocessing_model = preprocessing_pipeline.fit(train_df)
train_features = preprocessing_model.transform(train_df).select(LABEL_COL, 'features').persist(StorageLevel.MEMORY_AND_DISK)
test_features = preprocessing_model.transform(test_df).select(LABEL_COL, 'features').persist(StorageLevel.MEMORY_AND_DISK)
train_features.count(); test_features.count()
logger.info('[ML_PREPROCESSING] fit once on train_df and reused for every regression model')
end_stage('ML preprocessing fit')

## Distributed regression training and evaluation

In [ ]:
start_stage('Model training and evaluation')
models = {
    'LinearRegression': LinearRegression(labelCol=LABEL_COL, featuresCol='features', maxIter=(50 if TEST_MODE else 100), regParam=0.05, elasticNetParam=0.0),
    'RandomForestRegressor': RandomForestRegressor(labelCol=LABEL_COL, featuresCol='features', numTrees=(20 if TEST_MODE else 60), maxDepth=(8 if TEST_MODE else 12), seed=42),
    'GBTRegressor': GBTRegressor(labelCol=LABEL_COL, featuresCol='features', maxIter=(20 if TEST_MODE else 60), maxDepth=(5 if TEST_MODE else 6), seed=42),
}
evaluators = {metric: RegressionEvaluator(labelCol=LABEL_COL, predictionCol='prediction', metricName=metric) for metric in ['rmse', 'mae', 'r2']}
fitted_estimators, predictions_by_model, results = {}, {}, []
for model_name, estimator in models.items():
    logger.info('[ML] %s training started', model_name)
    training_started = time.perf_counter()
    fitted_estimator = estimator.fit(train_features)
    training_seconds = time.perf_counter() - training_started
    prediction_started = time.perf_counter()
    predictions = fitted_estimator.transform(test_features).persist(StorageLevel.MEMORY_AND_DISK)
    predictions.count()
    prediction_seconds = time.perf_counter() - prediction_started
    evaluation_started = time.perf_counter()
    metrics = {metric: float(evaluator.evaluate(predictions)) for metric, evaluator in evaluators.items()}
    evaluation_seconds = time.perf_counter() - evaluation_started
    fitted_estimators[model_name], predictions_by_model[model_name] = fitted_estimator, predictions
    result = {'model': model_name, **metrics, 'training_sec': training_seconds, 'prediction_sec': prediction_seconds, 'evaluation_sec': evaluation_seconds}
    results.append(result)
    logger.info('[EVALUATION] %s RMSE=%.4f MAE=%.4f R2=%.4f train=%.2fs predict=%.2fs evaluate=%.2fs', model_name, metrics['rmse'], metrics['mae'], metrics['r2'], training_seconds, prediction_seconds, evaluation_seconds)
end_stage('Model training and evaluation')

best_result = min(results, key=lambda result: result['rmse'])
best_model_name = best_result['model']
logger.info('[SUMMARY] Best model by lowest RMSE=%s; RMSE=%.4f MAE=%.4f R2=%.4f', best_model_name, best_result['rmse'], best_result['mae'], best_result['r2'])
print('================ MODEL COMPARISON ================')
for result in sorted(results, key=lambda item: item['rmse']): print(result)

In [ ]:
# ========= PATCH: Fix missing stage timer (training already completed) =========
import time
# Patch the missing stage_started entry so end_stage() doesn't throw KeyError
if 'Model training and evaluation' not in stage_started:
    total_train_time = sum(r.get('training_sec', 0) + r.get('prediction_sec', 0) + r.get('evaluation_sec', 0) for r in results)
    stage_started['Model training and evaluation'] = time.perf_counter() - total_train_time
    print(f'Patched stage timer. Estimated total training time: {total_train_time:.1f}s')

end_stage('Model training and evaluation')

best_result = min(results, key=lambda result: result['rmse'])
best_model_name = best_result['model']
logger.info('[SUMMARY] Best model by lowest RMSE=%s; RMSE=%.4f MAE=%.4f R2=%.4f', best_model_name, best_result['rmse'], best_result['mae'], best_result['r2'])
print('================ MODEL COMPARISON ================')
for result in sorted(results, key=lambda item: item['rmse']): print(result)


## Save distributed predictions and complete inference pipeline

In [ ]:
start_stage('Saving')
best_predictions_path = f'{PREDICTIONS_BASE_PATH}/{best_model_name}'
best_model_path = f'{MODEL_BASE_PATH}/{best_model_name}'
predictions_by_model[best_model_name].select(LABEL_COL, 'prediction').write.mode('overwrite').parquet(best_predictions_path)
complete_inference_pipeline = PipelineModel(stages=preprocessing_model.stages + [fitted_estimators[best_model_name]])
complete_inference_pipeline.write().overwrite().save(best_model_path)
logger.info('[SAVING] Predictions=%s; complete inference pipeline=%s', best_predictions_path, best_model_path)
end_stage('Saving')
ml_total = time.perf_counter() - pipeline_started
logger.info('[SUMMARY] ML total=%.2f sec; best=%s; predictions=%s; model=%s; log=%s', ml_total, best_model_name, best_predictions_path, best_model_path, LOG_PATH)
print('================ PART B BENCHMARK ================')
for stage, elapsed in benchmarks.items(): print(f'{stage:<36}{elapsed:>10.2f} sec')
print(f'ML TOTAL{ml_total:>33.2f} sec')
print(f'Best model: {best_model_name}; RMSE={best_result["rmse"]:.4f}; MAE={best_result["mae"]:.4f}; R2={best_result["r2"]:.4f}')
print(f'Predictions: {best_predictions_path}\nModel: {best_model_path}\nLog: {LOG_PATH}')
for predictions in predictions_by_model.values(): predictions.unpersist()
for frame in [processed_df, train_df, test_df, train_features, test_features]: frame.unpersist()
spark.stop()